# 🎾 Game Analytics: Streamlit Application
## Notebook 4 — Interactive Dashboard Code

This notebook contains the full Streamlit application code.

### How to Run
Streamlit apps **cannot** run inside a Jupyter notebook directly.
To launch the app:

1. Copy the code from the cell below into a file named `app.py`
2. Run: `streamlit run app.py`
3. The app opens at `http://localhost:8501`

### App Features (7 Pages):
1. **Homepage Dashboard** — Key statistics & overview charts
2. **competitions Analysis** — 7 competition queries with charts
3. **venues Analysis** — 7 venue queries with interactive filters
4. **Competitor Search & Filter** — Name search, rank/country/points filters
5. **Competitor Details Viewer** — Detailed stats for a selected competitor
6. **Country-Wise Analysis** — competitors & average points per country
7. **Leaderboards** — Top-ranked & highest-points tables + scatter plot

## Streamlit App Code (`app.py`)

> Copy this entire cell into `app.py` and run with `streamlit run app.py`

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text
import os

# ---------------------------------------------------------------------------
# Page configuration
# ---------------------------------------------------------------------------
st.set_page_config(
    page_title="Tennis Sportradar Analytics",
    page_icon="🎾",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ---------------------------------------------------------------------------
# Database configuration (PostgreSQL)
# ---------------------------------------------------------------------------
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "your_password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "tennis_db")
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# ---------------------------------------------------------------------------
# Database connection (cached)
# ---------------------------------------------------------------------------
@st.cache_resource
def get_engine():
    """Create a cached SQLAlchemy engine."""
    return create_engine(DATABASE_URL, pool_pre_ping=True)

@st.cache_data(ttl=300)
def run_query(query, _engine=None):
    """Execute a SQL query and return a DataFrame. Cached for 5 minutes."""
    eng = _engine or get_engine()
    with eng.connect() as conn:
        df = pd.read_sql(text(query), conn)
    return df

# ---------------------------------------------------------------------------
# Load all data once
# ---------------------------------------------------------------------------
@st.cache_data(ttl=300)
def load_all_data():
    """Load all tables into DataFrames for the app."""
    eng = get_engine()
    tables = {}
    queries_map = {
        "categories": "SELECT * FROM categories",
        "competitions": """
            SELECT c.*, cat.category_name
            FROM competitions c
            JOIN categories cat ON c.category_id = cat.category_id
        """,
        "complexes": "SELECT * FROM complexes",
        "venues": """
            SELECT v.*, cx.complex_name
            FROM venues v
            JOIN complexes cx ON v.complex_id = cx.complex_id
        """,
        "competitors": "SELECT * FROM competitors",
        "rankings": """
            SELECT cr.*, comp.name, comp.country, comp.country_code,
                   comp.abbreviation
            FROM competitor_rankings cr
            JOIN competitors comp ON cr.competitor_id = comp.competitor_id
        """,
    }
    for key, q in queries_map.items():
        try:
            tables[key] = run_query(q, eng)
        except Exception as e:
            st.error(f"Failed to load '{key}': {e}")
            tables[key] = pd.DataFrame()
    return tables

# ---------------------------------------------------------------------------
# Sidebar navigation
# ---------------------------------------------------------------------------
st.sidebar.title("🎾 Tennis Analytics")
st.sidebar.markdown("### Sportradar Data Explorer")

page = st.sidebar.radio(
    "Navigate",
    [
        "🏠 Homepage Dashboard",
        "🏆 competitions Analysis",
        "🏟️ venues Analysis",
        "🔍 Competitor Search & Filter",
        "👤 Competitor Details",
        "🌍 Country-Wise Analysis",
        "📊 Leaderboards",
    ],
)

# Load data
data = load_all_data()

# ---------------------------------------------------------------------------
# Page 1: Homepage Dashboard
# ---------------------------------------------------------------------------
if page == "🏠 Homepage Dashboard":
    st.title("🏠 Homepage Dashboard")
    st.markdown("---")

    competitors_df = data.get("competitors", pd.DataFrame())
    rankings_df = data.get("rankings", pd.DataFrame())
    competitions_df = data.get("competitions", pd.DataFrame())
    venues_df = data.get("venues", pd.DataFrame())

    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Total competitors", len(competitors_df))
    with col2:
        num_countries = competitors_df["country"].nunique() if not competitors_df.empty else 0
        st.metric("Countries Represented", num_countries)
    with col3:
        highest_points = int(rankings_df["points"].max()) if not rankings_df.empty else 0
        st.metric("Highest Points", highest_points)
    with col4:
        st.metric("Total competitions", len(competitions_df))

    st.markdown("---")

    col_left, col_right = st.columns(2)
    with col_left:
        st.subheader("competitions by Type")
        if not competitions_df.empty:
            type_counts = competitions_df["type"].value_counts().reset_index()
            type_counts.columns = ["Type", "Count"]
            fig = px.bar(type_counts, x="Type", y="Count", color="Type",
                         color_discrete_sequence=px.colors.qualitative.Set2)
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("No competition data available.")

    with col_right:
        st.subheader("Top 10 Countries by Competitor Count")
        if not competitors_df.empty:
            country_counts = competitors_df["country"].value_counts().head(10).reset_index()
            country_counts.columns = ["Country", "Count"]
            fig = px.bar(country_counts, x="Country", y="Count",
                         color="Count", color_continuous_scale="Viridis")
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("No competitor data available.")

    st.markdown("---")
    col_a, col_b = st.columns(2)
    with col_a:
        st.subheader("Total venues")
        st.metric("venues", len(venues_df))
    with col_b:
        st.subheader("Total complexes")
        complexes_df = data.get("complexes", pd.DataFrame())
        st.metric("complexes", len(complexes_df))

# ---------------------------------------------------------------------------
# Page 2: competitions Analysis
# ---------------------------------------------------------------------------
elif page == "🏆 competitions Analysis":
    st.title("🏆 competitions Analysis")
    st.markdown("---")
    competitions_df = data.get("competitions", pd.DataFrame())

    query_option = st.selectbox("Select Analysis", [
        "All competitions with category names",
        "Competition count per category",
        "Doubles competitions",
        "competitions in a specific category",
        "Parent & sub-competitions",
        "Competition type distribution by category",
        "Top-level competitions (no parent)",
    ])

    if query_option == "All competitions with category names":
        st.dataframe(competitions_df, use_container_width=True)
    elif query_option == "Competition count per category":
        if not competitions_df.empty:
            counts = competitions_df.groupby("category_name").size().reset_index(name="Count").sort_values("Count", ascending=False)
            st.dataframe(counts, use_container_width=True)
            fig = px.bar(counts, x="category_name", y="Count", color="Count", color_continuous_scale="Blues")
            st.plotly_chart(fig, use_container_width=True)
    elif query_option == "Doubles competitions":
        st.dataframe(competitions_df[competitions_df["type"] == "doubles"], use_container_width=True)
    elif query_option == "competitions in a specific category":
        if not competitions_df.empty:
            cat = st.selectbox("Choose category", competitions_df["category_name"].unique())
            st.dataframe(competitions_df[competitions_df["category_name"] == cat], use_container_width=True)
    elif query_option == "Parent & sub-competitions":
        if not competitions_df.empty and "parent_id" in competitions_df.columns:
            parents = competitions_df[competitions_df["parent_id"].notna()]
            merged = parents.merge(competitions_df[["competition_id", "competition_name"]], left_on="parent_id", right_on="competition_id", suffixes=("_child", "_parent"))
            result = merged[["competition_id_parent", "competition_name_parent", "competition_id_child", "competition_name_child"]].rename(columns={"competition_id_parent": "Parent ID", "competition_name_parent": "Parent Name", "competition_id_child": "Child ID", "competition_name_child": "Child Name"})
            st.dataframe(result, use_container_width=True)
    elif query_option == "Competition type distribution by category":
        if not competitions_df.empty:
            dist = competitions_df.groupby(["category_name", "type"]).size().reset_index(name="Count")
            fig = px.bar(dist, x="category_name", y="Count", color="type", barmode="stack", color_discrete_sequence=px.colors.qualitative.Set2)
            st.plotly_chart(fig, use_container_width=True)
            st.dataframe(dist, use_container_width=True)
    elif query_option == "Top-level competitions (no parent)":
        st.dataframe(competitions_df[competitions_df["parent_id"].isna()], use_container_width=True)

# ---------------------------------------------------------------------------
# Page 3: venues Analysis
# ---------------------------------------------------------------------------
elif page == "🏟️ venues Analysis":
    st.title("🏟️ venues Analysis")
    st.markdown("---")
    venues_df = data.get("venues", pd.DataFrame())

    query_option = st.selectbox("Select Analysis", [
        "All venues with complex names", "Venue count per complex",
        "venues in a specific country", "All venues and their timezones",
        "complexes with more than one venue", "venues grouped by country",
        "venues for a specific complex",
    ])

    if query_option == "All venues with complex names":
        st.dataframe(venues_df, use_container_width=True)
    elif query_option == "Venue count per complex":
        if not venues_df.empty:
            counts = venues_df.groupby("complex_name").size().reset_index(name="Venue Count").sort_values("Venue Count", ascending=False)
            st.dataframe(counts, use_container_width=True)
            fig = px.bar(counts, x="complex_name", y="Venue Count", color="Venue Count", color_continuous_scale="Greens")
            st.plotly_chart(fig, use_container_width=True)
    elif query_option == "venues in a specific country":
        if not venues_df.empty:
            country = st.selectbox("Choose country", sorted(venues_df["country_name"].unique()))
            st.dataframe(venues_df[venues_df["country_name"] == country], use_container_width=True)
    elif query_option == "All venues and their timezones":
        tz_cols = [c for c in ["venue_name", "city_name", "country_name", "timezone"] if c in venues_df.columns]
        st.dataframe(venues_df[tz_cols], use_container_width=True)
    elif query_option == "complexes with more than one venue":
        if not venues_df.empty:
            counts = venues_df.groupby("complex_name").size().reset_index(name="Venue Count")
            st.dataframe(counts[counts["Venue Count"] > 1], use_container_width=True)
    elif query_option == "venues grouped by country":
        if not venues_df.empty:
            st.dataframe(venues_df.sort_values(["country_name", "city_name"]), use_container_width=True)
    elif query_option == "venues for a specific complex":
        if not venues_df.empty:
            complex_name = st.selectbox("Choose complex", sorted(venues_df["complex_name"].unique()))
            st.dataframe(venues_df[venues_df["complex_name"] == complex_name], use_container_width=True)

# ---------------------------------------------------------------------------
# Page 4: Competitor Search & Filter
# ---------------------------------------------------------------------------
elif page == "🔍 Competitor Search & Filter":
    st.title("🔍 Competitor Search & Filter")
    st.markdown("---")
    rankings_df = data.get("rankings", pd.DataFrame())

    if rankings_df.empty:
        st.info("No competitor data available.")
    else:
        search_name = st.text_input("Search by competitor name:")
        col1, col2, col3 = st.columns(3)
        with col1:
            min_rank, max_rank = int(rankings_df["rank"].min()), int(rankings_df["rank"].max())
            rank_range = st.slider("Rank Range", min_rank, max_rank, (min_rank, max_rank))
        with col2:
            countries = sorted(rankings_df["country"].unique())
            selected_country = st.selectbox("Country", ["All"] + list(countries))
        with col3:
            max_points = int(rankings_df["points"].max())
            points_threshold = st.slider("Minimum Points", 0, max_points, 0)

        filtered = rankings_df.copy()
        if search_name:
            filtered = filtered[filtered["name"].str.contains(search_name, case=False, na=False)]
        filtered = filtered[(filtered["rank"] >= rank_range[0]) & (filtered["rank"] <= rank_range[1])]
        if selected_country != "All":
            filtered = filtered[filtered["country"] == selected_country]
        filtered = filtered[filtered["points"] >= points_threshold]
        st.markdown(f"**Showing {len(filtered)} competitors**")
        st.dataframe(filtered, use_container_width=True)

# ---------------------------------------------------------------------------
# Page 5: Competitor Details Viewer
# ---------------------------------------------------------------------------
elif page == "👤 Competitor Details":
    st.title("👤 Competitor Details Viewer")
    st.markdown("---")
    rankings_df = data.get("rankings", pd.DataFrame())

    if rankings_df.empty:
        st.info("No competitor data available.")
    else:
        selected_name = st.selectbox("Select a competitor", sorted(rankings_df["name"].unique()))
        competitor = rankings_df[rankings_df["name"] == selected_name].iloc[0]
        col1, col2, col3, col4 = st.columns(4)
        with col1: st.metric("Rank", int(competitor["rank"]))
        with col2: st.metric("Points", int(competitor["points"]))
        with col3: st.metric("Movement", int(competitor["movement"]))
        with col4: st.metric("competitions Played", int(competitor["competitions_played"]))
        st.markdown("---")
        st.markdown(f"**Country:** {competitor['country']}")
        st.markdown(f"**Country Code:** {competitor['country_code']}")
        st.markdown(f"**Abbreviation:** {competitor['abbreviation']}")

# ---------------------------------------------------------------------------
# Page 6: Country-Wise Analysis
# ---------------------------------------------------------------------------
elif page == "🌍 Country-Wise Analysis":
    st.title("🌍 Country-Wise Analysis")
    st.markdown("---")
    rankings_df = data.get("rankings", pd.DataFrame())

    if rankings_df.empty:
        st.info("No data available.")
    else:
        country_stats = rankings_df.groupby("country").agg(
            competitor_count=("name", "count"), total_points=("points", "sum"),
            avg_points=("points", "mean"), avg_rank=("rank", "mean")
        ).reset_index().sort_values("competitor_count", ascending=False)
        st.dataframe(country_stats, use_container_width=True)
        col1, col2 = st.columns(2)
        with col1:
            st.subheader("competitors per Country (Top 15)")
            fig = px.bar(country_stats.head(15), x="country", y="competitor_count", color="competitor_count", color_continuous_scale="Viridis")
            st.plotly_chart(fig, use_container_width=True)
        with col2:
            st.subheader("Average Points per Country (Top 15)")
            top_avg = country_stats.sort_values("avg_points", ascending=False).head(15)
            fig = px.bar(top_avg, x="country", y="avg_points", color="avg_points", color_continuous_scale="Cividis")
            st.plotly_chart(fig, use_container_width=True)

# ---------------------------------------------------------------------------
# Page 7: Leaderboards
# ---------------------------------------------------------------------------
elif page == "📊 Leaderboards":
    st.title("📊 Leaderboards")
    st.markdown("---")
    rankings_df = data.get("rankings", pd.DataFrame())

    if rankings_df.empty:
        st.info("No data available.")
    else:
        col1, col2 = st.columns(2)
        with col1:
            st.subheader("🏆 Top 10 by Rank")
            st.dataframe(rankings_df.nsmallest(10, "rank")[["rank", "name", "country", "points", "movement"]], use_container_width=True, hide_index=True)
        with col2:
            st.subheader("💎 Top 10 by Points")
            st.dataframe(rankings_df.nlargest(10, "points")[["rank", "name", "country", "points", "movement"]], use_container_width=True, hide_index=True)
        st.markdown("---")
        st.subheader("Rank vs Points Scatter")
        fig = px.scatter(rankings_df, x="rank", y="points", hover_name="name", color="country", size="points", title="Rank vs Points Distribution")
        st.plotly_chart(fig, use_container_width=True)


## How to Save & Run

```bash
# Option 1: Copy from this notebook
# Select all code in the cell above, paste into app.py, save

# Option 2: Use the included app.py file
streamlit run app.py
```

The app will open at `http://localhost:8501` in your browser.

### Environment Variables
Make sure these are set before running:
```bash
export DB_USER="postgres"
export DB_PASSWORD="your_password"
export DB_HOST="localhost"
export DB_PORT="5432"
export DB_NAME="tennis_db"
```